# PaceAlgo ML — Notebook 13: Cross-Asset Generalization (Phase B Research Platform)

**Forschungsfrage:** Wie gut generalisiert unser FX-trainiertes Modell auf andere Asset-Klassen? Welche Features sind universell, welche asset-spezifisch?

**Strategy-Locks (siehe [ANN-006](../docs/decisions/ANN-006-robustness-first-mantra.md)):**
1. Generalisierung > Maximierung eines einzelnen PF-Werts
2. Robustheit > Benchmark-Chasing
3. Konsistenz > Cherry-Picking
4. Gute UX + ehrliches Backtesting > Marketing-Zahlen

**6 Hypothesen** (siehe [research/asset_generalization.md](../research/asset_generalization.md)) — NB13 testet H1–H6 deterministisch via Schwellen aus `core.config.PHASE_B_THRESHOLDS`.

**Notebook-Struktur:**
- **Section 0:** Config + Experiment Registry
- **Section 1:** Data Loading + Inventory Check
- **Section 2:** Feature Engineering (modular per Asset-Group)
- **Section 3:** Labeling Check
- **Section 4:** Walk-Forward Split Builder
- **Section 5:** Model Training Loop (Asset-Group × TF × Model)
- **Section 6:** SHAP Analysis (Global / per Asset-Class / per TF / Stability)
- **Section 7:** Cross-Asset Generalization Matrix
- **Section 8:** Timeframe Comparison
- **Section 9:** Architecture Decision Engine
- **Section 10:** Result Persistence (`/results/nb13/`)
- **Section 11:** Final Verdict
- **Section 12:** Auto-Push Results to GitHub

## Section 0 — Config + Experiment Registry

Zentrale Konfiguration. Alles was zwischen Runs variieren könnte (Asset-Scope, TFs, Modelle, Seeds) lebt hier.

**EXPERIMENT_ID:** dient als Trace-Marker, wird in jede /results/-Datei eingebaut.

In [ ]:
import sys, os, json, time, subprocess
from pathlib import Path
from datetime import datetime, timezone

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    os.chdir(PROJECT_ROOT)
    # Refresh core code from GitHub (latest)
    !rm -rf /tmp/pace-algo
    !git clone -q https://github.com/ecoNC/pace-algo.git /tmp/pace-algo
    !cp -rf /tmp/pace-algo/core/* {PROJECT_ROOT}/core/
    print('Core code synced from GitHub')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)

sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]

# Reproducibility
RANDOM_SEED = 42
import numpy as np
np.random.seed(RANDOM_SEED)

# Run identity
RUN_DATE      = datetime.now(timezone.utc).strftime('%Y-%m-%d')
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')
try:
    GIT_COMMIT = subprocess.check_output(['git', '-C', PROJECT_ROOT, 'rev-parse', '--short', 'HEAD'], text=True).strip()
except Exception:
    GIT_COMMIT = 'unknown'
EXPERIMENT_ID = f'nb13_{RUN_TIMESTAMP}_{GIT_COMMIT}'

# Result paths (versioned per RUN_DATE)
RESULTS_DIR = Path(PROJECT_ROOT) / 'results' / 'nb13'
for sub in ('metrics', 'shap', 'predictions', 'summaries', 'plots', 'config_snapshots'):
    (RESULTS_DIR / sub).mkdir(parents=True, exist_ok=True)
MODELS_DIR_NB13 = Path(PROJECT_ROOT) / 'artifacts' / 'models' / 'nb13'
MODELS_DIR_NB13.mkdir(parents=True, exist_ok=True)

print(f'EXPERIMENT_ID: {EXPERIMENT_ID}')
print(f'RUN_DATE:      {RUN_DATE}')
print(f'GIT_COMMIT:    {GIT_COMMIT}')
print(f'RESULTS_DIR:   {RESULTS_DIR}')
print(f'MODELS_DIR:    {MODELS_DIR_NB13}')
print(f'RANDOM_SEED:   {RANDOM_SEED}')

In [ ]:
# Install/import ML stack
!pip install -q lightgbm xgboost catboost shap 2>&1 | tail -1

import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm
import shap
import warnings
warnings.filterwarnings('ignore')

# Pull NB13-specific config from core.config
from core.config import (
    DATA_RAW, DATA_PROCESSED, ARTIFACTS_REPORTS,
    FX_TRAIN_SYMBOLS, FX_HOLDOUT_SYMBOLS, FX_SYMBOLS,
    CRYPTO_SYMBOLS, METAL_SYMBOLS, INDEX_SYMBOLS_FUTURE,
    ASSET_GROUPS,
    PRIMARY_TIMEFRAMES, HTF_CONTEXT_TIMEFRAMES,
    TRAIN_END, VAL_END, TEST_END,
    PHASE_B_THRESHOLDS,
)
from core.features import (
    compute_features, attach_macro, attach_htf_context,
    compute_smc_features, compute_session_features, compute_htf_interactions,
)
from core.features.engineer import atr as atr_fn
from core.train import walk_forward_split, binary_label_for_long

print('Lib versions:')
print(f'  LightGBM: {lgb.__version__}')
print(f'  XGBoost:  {xgb.__version__}')
import catboost
print(f'  CatBoost: {catboost.__version__}')
print(f'  SHAP:     {shap.__version__}')

In [ ]:
# Experiment Registry — control what runs in this session
# Default: research mode (frisch trainieren, fx_train pool only — memory-safe on Colab Free)

TRAIN_FRESH      = True   # immer frisch trainieren — wissenschaftlich sauber (siehe Nico-Lock)
LOAD_CACHE       = False  # nur für Fast-Iteration setzen, NIE silent

# Welche Experimente laufen in diesem Run?
# WICHTIG (Memory): Experiment D (universal pool) braucht ~6 GB RAM für 5m+15m und sprengt Colab Free.
# Default = False. Aktiviere nur mit Colab Pro / High-RAM-Runtime.
EXPERIMENTS_TO_RUN = {
    'A_fx_to_fx_holdout':  True,   # FX-trained -> GBPUSD/AUDUSD/USDCHF
    'B_fx_to_crypto':      True,   # FX-trained -> BTC/ETH/SOL/BNB/ADA
    'C_fx_to_indices':     False,  # REQUIRES Polygon — enable wenn polygon_fetcher.py existiert
    'D_universal_trained': False,  # Memory-heavy — separates NB13b oder High-RAM-Runtime
    'E_asset_cluster':     False,  # Cluster-Modelle — separates NB13c (komplex)
}

# Modelle pro Experiment
MODELS_TO_TRAIN = ['LightGBM']  # starten schlank; XGB/CatBoost wenn Pattern verstanden

# TFs pro Experiment — Default ist alle, kann reduziert werden für smoke-tests
TIMEFRAMES_USED = PRIMARY_TIMEFRAMES  # ['5m', '15m', '30m', '1h']

# Snapshot der gesamten Run-Config — wird mit jedem Result persisted
RUN_CONFIG = {
    'experiment_id':       EXPERIMENT_ID,
    'run_date':            RUN_DATE,
    'git_commit':          GIT_COMMIT,
    'random_seed':         RANDOM_SEED,
    'train_fresh':         TRAIN_FRESH,
    'load_cache':          LOAD_CACHE,
    'experiments_to_run':  EXPERIMENTS_TO_RUN,
    'models_to_train':     MODELS_TO_TRAIN,
    'timeframes_used':     TIMEFRAMES_USED,
    'fx_train_symbols':    FX_TRAIN_SYMBOLS,
    'fx_holdout_symbols':  FX_HOLDOUT_SYMBOLS,
    'crypto_symbols':      CRYPTO_SYMBOLS,
    'metal_symbols':       METAL_SYMBOLS,
    'index_symbols':       INDEX_SYMBOLS_FUTURE if EXPERIMENTS_TO_RUN['C_fx_to_indices'] else [],
    'train_end':           str(TRAIN_END),
    'val_end':             str(VAL_END),
    'phase_b_thresholds':  PHASE_B_THRESHOLDS,
    'lib_versions': {
        'lightgbm': lgb.__version__,
        'xgboost':  xgb.__version__,
        'catboost': catboost.__version__,
        'shap':     shap.__version__,
    },
}

config_snapshot_path = RESULTS_DIR / 'config_snapshots' / f'{EXPERIMENT_ID}_config.json'
with open(config_snapshot_path, 'w') as f:
    json.dump(RUN_CONFIG, f, indent=2, default=str)
print(f'Config snapshot → {config_snapshot_path.relative_to(PROJECT_ROOT)}')
print(f'\nActive experiments: {[k for k, v in EXPERIMENTS_TO_RUN.items() if v]}')
print(f'Models: {MODELS_TO_TRAIN}')
print(f'Timeframes: {TIMEFRAMES_USED}')

## Section 1 — Data Loading + Inventory Check

Prüft welche Symbole × TFs an Roh-OHLCV-Daten verfügbar sind. Wenn was fehlt: klarer Fail mit Hinweis welcher Fetcher zu starten ist.

In [ ]:
if IS_COLAB:
    DATA_RAW_PATH = Path(PROJECT_ROOT) / 'data_cache' / 'raw'
else:
    DATA_RAW_PATH = DATA_RAW
DATA_RAW_PATH.mkdir(parents=True, exist_ok=True)

# Welche Symbole brauchen wir?
needed_symbols = set()
if EXPERIMENTS_TO_RUN['A_fx_to_fx_holdout']:
    needed_symbols.update(FX_TRAIN_SYMBOLS + FX_HOLDOUT_SYMBOLS)
if EXPERIMENTS_TO_RUN['B_fx_to_crypto']:
    needed_symbols.update(FX_TRAIN_SYMBOLS + CRYPTO_SYMBOLS)
if EXPERIMENTS_TO_RUN['C_fx_to_indices']:
    needed_symbols.update(FX_TRAIN_SYMBOLS + INDEX_SYMBOLS_FUTURE)
if EXPERIMENTS_TO_RUN['D_universal_trained']:
    needed_symbols.update(FX_TRAIN_SYMBOLS + CRYPTO_SYMBOLS + METAL_SYMBOLS + FX_HOLDOUT_SYMBOLS)

# HTF-Context wird für alle Symbole × alle HTF_CONTEXT_TIMEFRAMES gebraucht
needed_tfs_per_symbol = set(TIMEFRAMES_USED) | set(HTF_CONTEXT_TIMEFRAMES)

# Inventory check
missing = []
present = []
for s in sorted(needed_symbols):
    for tf in sorted(needed_tfs_per_symbol):
        p = DATA_RAW_PATH / f'{s}_{tf}.parquet'
        if p.exists():
            present.append((s, tf))
        else:
            missing.append((s, tf))

print(f'Data inventory check:')
print(f'  Needed: {len(needed_symbols)} symbols × {len(needed_tfs_per_symbol)} TFs = {len(needed_symbols) * len(needed_tfs_per_symbol)} files')
print(f'  Present: {len(present)} files')
print(f'  Missing: {len(missing)} files')

if missing:
    print(f'\n⚠️  Missing OHLCV files — fetch these first:')
    missing_by_symbol = {}
    for s, tf in missing:
        missing_by_symbol.setdefault(s, []).append(tf)
    for s, tfs in sorted(missing_by_symbol.items()):
        print(f'    {s}: {tfs}')
    print(f'\nAction needed:')
    print(f'  - FX/Metals: run NB01 Dukascopy-fetcher with extended symbol list')
    print(f'  - Crypto:   run NB01 KuCoin-fetcher with extended symbol list')
    print(f'  - Indices:  REQUIRES Polygon-fetcher (not built yet) — set EXPERIMENTS_TO_RUN["C_fx_to_indices"] = False')
    raise SystemExit('Data not ready. Run NB01 (extended) and come back.')
else:
    print(f'\n✅ All needed OHLCV files present.')

## Section 2 — Feature Engineering (modular per Asset-Group)

Build extended features (base + HTF + session + interactions) für alle Symbole × TFs. Reused von NB12-Pattern, aber generalisiert auf größeren Symbol-Pool.

**Wichtig:** Labels (`labels_{symbol}_{tf}_R1.5.parquet`) müssen vorher von NB04 generiert sein.

In [ ]:
R_VALUE = 1.5
WIN_R   = 1.5
LOSS_R  = 1.0

if IS_COLAB:
    DATA_EXT = Path('/content/processed_v2')
    DATA_PROCESSED_LOCAL = Path('/content/processed')
    EXT_DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed_v2'
    DRIVE_BACKUP_PROCESSED = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    
    DATA_EXT.mkdir(parents=True, exist_ok=True)
    DATA_PROCESSED_LOCAL.mkdir(parents=True, exist_ok=True)
    EXT_DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
    
    # Restore labels + extended features from Drive backup if local is empty
    if len(list(DATA_PROCESSED_LOCAL.glob('labels_*.parquet'))) == 0 and DRIVE_BACKUP_PROCESSED.exists():
        print('Restoring labels from Drive backup...')
        !rsync -ah {DRIVE_BACKUP_PROCESSED}/ {DATA_PROCESSED_LOCAL}/
    if len(list(DATA_EXT.glob('*_extended.parquet'))) == 0 and EXT_DRIVE_BACKUP.exists():
        print('Restoring extended features from Drive backup...')
        !rsync -ah {EXT_DRIVE_BACKUP}/ {DATA_EXT}/
else:
    DATA_EXT = DATA_PROCESSED.parent / 'processed_v2'
    DATA_PROCESSED_LOCAL = DATA_PROCESSED
DATA_EXT.mkdir(parents=True, exist_ok=True)

def load_ohlcv_raw(symbol, tf):
    p = DATA_RAW_PATH / f'{symbol}_{tf}.parquet'
    return pd.read_parquet(p) if p.exists() else None

def load_ext(sym, tf):
    p = DATA_EXT / f'{sym}_{tf}_extended.parquet'
    return pd.read_parquet(p) if p.exists() else None

def build_extended_features(symbol, tf):
    """Compute extended features for one (symbol, tf) pair, including labels."""
    ohlcv = load_ohlcv_raw(symbol, tf)
    if ohlcv is None or ohlcv.empty:
        return None
    # Base features
    base = compute_features(ohlcv)
    # HTF context (1h + 4h)
    htf_data = {}
    for htf in HTF_CONTEXT_TIMEFRAMES:
        d = load_ohlcv_raw(symbol, htf)
        htf_data[htf] = compute_features(d) if (d is not None and not d.empty) else pd.DataFrame()
    base = attach_htf_context(base, htf_data.get('1h', pd.DataFrame()), htf_data.get('4h', pd.DataFrame()))
    # Macro (deprecated but kept for compatibility)
    macro_path = DATA_RAW_PATH / 'macro_daily.parquet'
    macro = pd.read_parquet(macro_path) if macro_path.exists() else pd.DataFrame()
    base = attach_macro(base, macro)
    # Additional feature groups
    atr14 = atr_fn(ohlcv['high'], ohlcv['low'], ohlcv['close'], 14).values
    ema_align = base['ema_alignment'].fillna(0).values if 'ema_alignment' in base.columns else np.zeros(len(base))
    smc = compute_smc_features(ohlcv, atr14, ema_align)
    sess = compute_session_features(ohlcv, atr14)
    inter = compute_htf_interactions(base)
    ext = pd.concat([base, smc, sess, inter], axis=1)
    # Attach labels
    label_path = DATA_PROCESSED_LOCAL / f'labels_{symbol}_{tf}_R{R_VALUE}.parquet'
    if not label_path.exists():
        return None
    labels = pd.read_parquet(label_path)
    ext = ext.join(labels[['label']], how='inner')
    ext['symbol'] = symbol
    ext['timeframe'] = tf
    return ext

# Build missing extended files
expected = [(s, tf) for s in needed_symbols for tf in TIMEFRAMES_USED]
missing_ext = [(s, tf) for s, tf in expected if not (DATA_EXT / f'{s}_{tf}_extended.parquet').exists()]
print(f'Extended features: {len(expected) - len(missing_ext)} present, {len(missing_ext)} to build')

if missing_ext:
    for s, tf in tqdm(missing_ext, desc='Building extended'):
        ext = build_extended_features(s, tf)
        if ext is None:
            print(f'  SKIP {s} {tf}: data or labels missing')
            continue
        out = DATA_EXT / f'{s}_{tf}_extended.parquet'
        ext.to_parquet(out, compression='zstd')

if IS_COLAB:
    print('Syncing extended features to Drive backup...')
    !rsync -ah {DATA_EXT}/ {EXT_DRIVE_BACKUP}/

n_now = len(list(DATA_EXT.glob('*_extended.parquet')))
print(f'\nExtended feature files now: {n_now}')

## Section 3 — Labeling Check + Class Balance Report

Schneller Health-Check: stimmen die Label-Verteilungen über Asset-Klassen? Wenn z.B. Crypto-Labels nur 5% positive haben, ist das Modell auf Crypto strukturell anders gefordert als auf FX.

In [ ]:
def label_balance_for(symbols, tfs):
    rows = []
    for s in symbols:
        for tf in tfs:
            df = load_ext(s, tf)
            if df is None or df.empty:
                continue
            y = binary_label_for_long(df['label']).values
            rows.append({
                'symbol': s, 'timeframe': tf,
                'n': int(len(y)),
                'pos_rate': float(y.mean()),
            })
    return pd.DataFrame(rows)

balance_df = label_balance_for(sorted(needed_symbols), TIMEFRAMES_USED)
print('=== Label balance (positive rate) per symbol × TF ===')
if not balance_df.empty:
    pivot = balance_df.pivot_table(index='symbol', columns='timeframe', values='pos_rate').round(3)
    display(pivot)
    print('\n=== Bar count per symbol × TF ===')
    pivot_n = balance_df.pivot_table(index='symbol', columns='timeframe', values='n').round(0).astype(int)
    display(pivot_n)
else:
    print('No extended data loaded — check Section 2.')

## Section 4 — Walk-Forward Split Builder

Strikt zeitbasiert (TRAIN < VAL < TEST). Identische Cutoffs über alle Asset-Gruppen für Fairness.

In [ ]:
def build_pool(symbols, tfs, dtype='float32'):
    """Stack extended features for given symbols × tfs into one DataFrame.

    dtype='float32' halbiert Memory-Footprint gegenüber float64. Reicht für 30-tree LGBM.
    """
    frames = []
    for s in symbols:
        for tf in tfs:
            d = load_ext(s, tf)
            if d is not None and not d.empty:
                # Downcast nur die Feature-Spalten, nicht 'label' (int) oder 'symbol'/'timeframe' (str)
                float_cols = d.select_dtypes(include=['float64']).columns
                if len(float_cols) > 0:
                    d = d.astype({c: dtype for c in float_cols}, copy=False)
                frames.append(d)
    if not frames:
        return pd.DataFrame()
    pool = pd.concat(frames, axis=0).sort_index()
    del frames
    return pool

def split_and_report(pool, label_name='POOL'):
    if pool.empty:
        print(f'{label_name}: empty')
        return None, None, None
    train_df, val_df, test_df = walk_forward_split(pool, TRAIN_END, VAL_END)
    print(f'{label_name:30s} train={len(train_df):>8,}  val={len(val_df):>7,}  test={len(test_df):>8,}')
    return train_df, val_df, test_df

# Load feature set from NB11-winner (27 features) — same as NB12 baseline
FEATURE_COLS_NB11_WINNER = [
    'hour_sin', 'dist_to_swing_low_atr', 'htf_1h_rsi_14', 'realized_vol_20',
    'atr_percentile_100', 'atr_pct', 'dist_to_swing_high_atr', 'volume_z_score',
    'ema_20_slope_atr', 'hour_cos', 'momentum_composite', 'rvol_20',
    'adx_14', 'ema_20_dist_atr', 'htf_1h_atr_percentile_100',
    'htf_ltf_agree_bull', 'htf_ltf_agree_bear', 'htf_ltf_counter_trend',
    'htf_ltf_alignment_score', 'ltf_rsi_minus_htf_rsi',
    'both_rsi_oversold', 'both_rsi_overbought', 'vol_pct_diff_htf',
    'both_high_vol', 'both_low_vol', 'pullback_in_bull', 'pullback_in_bear',
]
print(f'Using {len(FEATURE_COLS_NB11_WINNER)} features from NB11-winner config')

## Section 5 — Model Training Loop (Asset-Group × TF)

**Training-Strategie:** Wir trainieren ein Modell pro (training_pool, TF). Dann inferenzen wir das Modell auf jedem Asset einzeln.

**Training-Pools dieses Runs:**
- `fx_train` = `FX_TRAIN_SYMBOLS` (Experimente A, B, C nutzen das gleiche Modell)
- `universal` = `FX_TRAIN + CRYPTO + GOLD` (Experiment D)

**Modelle pro Pool × TF:** je nach `MODELS_TO_TRAIN`.

In [ ]:
import gc

def train_lightgbm(X_tr, y_tr, X_vl, y_vl):
    params = {
        'objective': 'binary', 'metric': 'binary_logloss',
        'num_leaves': 7, 'max_depth': 3, 'min_data_in_leaf': 200,
        'learning_rate': 0.05, 'num_iterations': 30, 'lambda_l2': 0.5,
        'feature_fraction': 0.85, 'bagging_fraction': 0.85, 'bagging_freq': 5,
        'is_unbalance': True, 'verbose': -1, 'n_jobs': -1,
        'seed': RANDOM_SEED, 'deterministic': True,
    }
    td = lgb.Dataset(X_tr, label=y_tr)
    vd = lgb.Dataset(X_vl, label=y_vl, reference=td)
    booster = lgb.train(params, td, num_boost_round=30, valid_sets=[vd],
                         callbacks=[lgb.log_evaluation(period=0)])
    # Free Dataset memory immediately
    del td, vd
    gc.collect()
    return booster

def predict_proba(model, model_name, X):
    if model_name == 'LightGBM':
        return model.predict(X)
    elif model_name in ('XGBoost', 'CatBoost'):
        return model.predict_proba(X)[:, 1]
    raise ValueError(model_name)

# Define training pools for THIS run
TRAINING_POOLS = {}
if any([EXPERIMENTS_TO_RUN['A_fx_to_fx_holdout'],
        EXPERIMENTS_TO_RUN['B_fx_to_crypto'],
        EXPERIMENTS_TO_RUN['C_fx_to_indices']]):
    TRAINING_POOLS['fx_train'] = FX_TRAIN_SYMBOLS
if EXPERIMENTS_TO_RUN['D_universal_trained']:
    TRAINING_POOLS['universal'] = FX_TRAIN_SYMBOLS + CRYPTO_SYMBOLS + METAL_SYMBOLS

# Cache der VAL-Probas für Section 7 (vermeidet 2. build_pool für jedes Modell)
val_probas_cache = {}   # val_probas_cache[(pool_name, tf, model)] = np.ndarray
val_cutoffs_cache = {}  # val_cutoffs_cache[(pool_name, tf, model)] = {'Standard':x,'High':y,'Premium':z}

def _cutoffs_from_val(val_probas, percentages=(10, 3, 1)):
    vs = np.sort(val_probas)[::-1]
    return {
        'Standard': float(vs[max(1, int(len(vs) * percentages[0] / 100) - 1)]),
        'High':     float(vs[max(1, int(len(vs) * percentages[1] / 100) - 1)]),
        'Premium':  float(vs[max(1, int(len(vs) * percentages[2] / 100) - 1)]),
    }

trained = {}   # trained[pool_name][tf][model_name] = (model, available_features)
for pool_name, pool_symbols in TRAINING_POOLS.items():
    trained[pool_name] = {}
    for tf in TIMEFRAMES_USED:
        print(f'\n--- {pool_name} / {tf} ---')
        pool = build_pool(pool_symbols, [tf])
        if pool.empty:
            print(f'SKIP: no data')
            continue
        avail = [f for f in FEATURE_COLS_NB11_WINNER if f in pool.columns]
        pool_c = pool.dropna(subset=avail + ['label'])
        train_df, val_df, _ = walk_forward_split(pool_c, TRAIN_END, VAL_END)
        # Free the full pool — we only need train/val arrays now
        del pool, pool_c
        gc.collect()

        # Convert to numpy and free DataFrames where possible
        X_tr = train_df[avail].values.astype('float32', copy=False)
        y_tr = binary_label_for_long(train_df['label']).values
        X_vl = val_df[avail].values.astype('float32', copy=False)
        y_vl = binary_label_for_long(val_df['label']).values
        n_train = len(train_df); n_val = len(val_df)
        del train_df, val_df
        gc.collect()

        print(f'  train={n_train:,}  val={n_val:,}  features={len(avail)}')
        trained[pool_name][tf] = {}
        for mname in MODELS_TO_TRAIN:
            if mname == 'LightGBM':
                m = train_lightgbm(X_tr, y_tr, X_vl, y_vl)
            else:
                raise NotImplementedError(f'Model {mname} not yet wired up (only LightGBM in MVP)')
            trained[pool_name][tf][mname] = (m, avail)
            # Persist model snapshot
            mpath = MODELS_DIR_NB13 / f'{pool_name}_{tf}_{mname}_{RUN_DATE}.txt'
            if mname == 'LightGBM':
                m.save_model(str(mpath))
            # Cache VAL probas + cutoffs (used in Section 7, avoids rebuild_pool again)
            vp = predict_proba(m, mname, X_vl)
            val_probas_cache[(pool_name, tf, mname)] = vp
            val_cutoffs_cache[(pool_name, tf, mname)] = _cutoffs_from_val(vp)
            print(f'  {mname} done → {mpath.relative_to(PROJECT_ROOT)}')

        # Explicit cleanup before next (pool, tf) iteration
        del X_tr, y_tr, X_vl, y_vl
        gc.collect()

print('\n--- Training Loop Done ---')
print(f'Trained models: {sum(len(by_tf) for by_tf in trained.values())} (pool, tf, model) tuples')

## Section 7 — Cross-Asset Generalization Matrix

Inferenz aller trainierten Modelle auf jedem Asset einzeln. Liefert die zentrale Tabelle: **Pool x TF x Test-Asset → Premium-PF**.

In [ ]:
def tier_metrics(labels):
    wins = int((labels == 1).sum())
    losses = int((labels == -1).sum())
    neutrals = int((labels == 0).sum())
    total = wins + losses + neutrals
    pf = (wins * WIN_R) / (losses * LOSS_R) if losses > 0 else (float('inf') if wins > 0 else 0.0)
    wr = wins / (wins + losses) if (wins + losses) > 0 else 0.0
    expR = (wins * WIN_R - losses * LOSS_R) / total if total > 0 else 0.0
    return {'n': total, 'wins': wins, 'losses': losses, 'wr': wr, 'pf': pf, 'expR': expR}

val_end_ts = pd.Timestamp(VAL_END)
if val_end_ts.tz is None:
    val_end_ts = val_end_ts.tz_localize('UTC')

rows = []
for pool_name, by_tf in trained.items():
    for tf, by_model in by_tf.items():
        for mname, (model, avail) in by_model.items():
            # Use cached VAL cutoffs from training loop (avoids rebuild_pool — saves memory)
            cutoffs = val_cutoffs_cache[(pool_name, tf, mname)]
            # Inferenz auf jedem Symbol einzeln (OOS-Window only: index >= VAL_END)
            for test_sym in sorted(needed_symbols):
                test_df = load_ext(test_sym, tf)
                if test_df is None or test_df.empty:
                    continue
                test_df = test_df.dropna(subset=avail + ['label'])
                test_df = test_df[test_df.index >= val_end_ts]
                if test_df.empty:
                    continue
                # Identify asset class
                asset_class = next((k for k, v in ASSET_GROUPS.items() if test_sym in v), 'unknown')
                X_test = test_df[avail].values.astype('float32', copy=False)
                probas = predict_proba(model, mname, X_test)
                labels_arr = test_df['label'].values
                del test_df, X_test
                for tier, cutoff in cutoffs.items():
                    mask = probas >= cutoff
                    sub = labels_arr[mask]
                    m = tier_metrics(sub)
                    rows.append({
                        'pool': pool_name, 'tf': tf, 'model': mname,
                        'test_symbol': test_sym, 'asset_class': asset_class,
                        'tier': tier, 'cutoff': cutoff, **m,
                    })
                del probas, labels_arr
                gc.collect()

matrix_df = pd.DataFrame(rows)
print(f'Generated {len(matrix_df)} (pool, tf, model, symbol, tier) rows')

# Premium-Tier-Pivot per pool × tf × asset_class (mean PF)
premium = matrix_df[matrix_df['tier'] == 'Premium'].copy()
if not premium.empty:
    print('\n=== Premium PF mean by pool × asset_class (over all TFs + symbols) ===')
    pivot = premium.pivot_table(index='pool', columns='asset_class', values='pf', aggfunc='mean').round(3)
    display(pivot)
    print('\n=== Premium PF mean by pool × TF ===')
    pivot_tf = premium.pivot_table(index='pool', columns='tf', values='pf', aggfunc='mean').round(3)
    display(pivot_tf)

## Section 8 — Timeframe Comparison

Welche TFs liefern stabilste OOS-Ergebnisse über Asset-Klassen?

In [ ]:
tf_summary_rows = []
for tf in TIMEFRAMES_USED:
    sub = premium[(premium['tf'] == tf) & (premium['pool'] == 'fx_train')]
    if sub.empty:
        continue
    finite = sub[~sub['pf'].isin([np.inf, -np.inf])]
    mean_pf = float(finite['pf'].mean())
    min_pf  = float(finite['pf'].min())
    max_pf  = float(finite['pf'].max())
    cv      = float(finite['pf'].std() / finite['pf'].mean()) if finite['pf'].mean() != 0 else float('nan')
    total_trades = int(sub['n'].sum())
    tf_summary_rows.append({
        'tf': tf, 'mean_pf': mean_pf, 'min_pf': min_pf, 'max_pf': max_pf,
        'cv': cv, 'total_trades': total_trades,
    })
tf_summary = pd.DataFrame(tf_summary_rows).round(3)
print('=== TF comparison (fx_train pool, Premium tier, mean across all test symbols) ===')
display(tf_summary)

## Section 6 — SHAP Analysis (Global / per Asset-Class / per TF / Stability)

**4 Ebenen** (Nico-Lock):
1. Global SHAP — alle Assets/TFs gepoolt
2. SHAP per Asset-Klasse — welche Features funktionieren universell?
3. SHAP per TF — welche Features sind TF-spezifisch?
4. SHAP Consistency Score — Stabilität über Klassen × TFs

**MVP-Variante:** SHAP wird auf einer Sample-Untermenge berechnet (5000 rows per group) für Speed. Volle SHAP kommt in NB13b oder NB14.

In [ ]:
SHAP_SAMPLE_SIZE = 5000  # per group/tf — reduzieren wenn Memory eng

def shap_for(model, model_name, X_sample, feature_names):
    if model_name == 'LightGBM':
        explainer = shap.TreeExplainer(model)
    elif model_name == 'XGBoost':
        explainer = shap.TreeExplainer(model)
    elif model_name == 'CatBoost':
        explainer = shap.TreeExplainer(model)
    else:
        return None
    shap_vals = explainer.shap_values(X_sample)
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1] if len(shap_vals) > 1 else shap_vals[0]
    abs_mean = np.abs(shap_vals).mean(axis=0)
    return dict(zip(feature_names, abs_mean.tolist()))

# SHAP per (pool, tf, asset_class)
shap_rows = []
for pool_name, by_tf in trained.items():
    if pool_name != 'fx_train':  # SHAP focus on fx_train pool (cross-asset generalization)
        continue
    for tf, by_model in by_tf.items():
        for mname, (model, avail) in by_model.items():
            for asset_class, syms in ASSET_GROUPS.items():
                test_syms = [s for s in syms if s in needed_symbols]
                if not test_syms:
                    continue
                pool = build_pool(test_syms, [tf]).dropna(subset=avail + ['label'])
                if pool.empty or len(pool) < 100:
                    continue
                sample = pool.sample(min(SHAP_SAMPLE_SIZE, len(pool)), random_state=RANDOM_SEED)
                shap_dict = shap_for(model, mname, sample[avail].values, avail)
                if shap_dict is None:
                    continue
                for feat, val in shap_dict.items():
                    shap_rows.append({
                        'pool': pool_name, 'tf': tf, 'model': mname,
                        'asset_class': asset_class, 'feature': feat, 'mean_abs_shap': val,
                    })

shap_df = pd.DataFrame(shap_rows)
if not shap_df.empty:
    print(f'SHAP rows: {len(shap_df)}')
    print('\n=== Top 10 features by global mean |SHAP| (fx_train, all TFs/classes pooled) ===')
    global_top = shap_df.groupby('feature')['mean_abs_shap'].mean().sort_values(ascending=False).head(10).round(4)
    display(global_top)
    print('\n=== SHAP rank per feature per asset_class (Top 15) ===')
    ranked = shap_df.groupby(['asset_class', 'feature'])['mean_abs_shap'].mean().reset_index()
    ranked['rank'] = ranked.groupby('asset_class')['mean_abs_shap'].rank(ascending=False, method='dense').astype(int)
    pivot_rank = ranked.pivot_table(index='feature', columns='asset_class', values='rank', aggfunc='min')
    pivot_rank = pivot_rank.assign(rank_std=pivot_rank.std(axis=1)).sort_values('rank_std', ascending=True)
    display(pivot_rank.head(15).round(2))
    print('\n(rank_std klein = Feature stabil über Asset-Klassen — universeller Edge)')

## Section 9 — Architecture Decision Engine

Auto-Scoring der Hypothesen H1–H6 mit Schwellen aus `core.config.PHASE_B_THRESHOLDS`.

**Erwartete Output:**
- H1: erfüllt / nicht erfüllt + Mean-PF-Wert
- H5/H6: Anzahl Asset-Klassen die Schwelle überschreiten
- Empfehlung: Variante A / B / C für Phase D

In [ ]:
verdict = {}
thr = PHASE_B_THRESHOLDS

# H1 — Mean PF über Asset-Klassen
if EXPERIMENTS_TO_RUN['A_fx_to_fx_holdout'] or EXPERIMENTS_TO_RUN['B_fx_to_crypto']:
    fx_pool_premium = premium[(premium['pool'] == 'fx_train') & (premium['model'] == 'LightGBM')]
    if not fx_pool_premium.empty:
        finite = fx_pool_premium[~fx_pool_premium['pf'].isin([np.inf, -np.inf])]
        # Aggregate per asset_class then mean
        class_means = finite.groupby('asset_class')['pf'].mean()
        verdict['h1_mean_pf']         = float(class_means.mean())
        verdict['h1_min_pf_per_class'] = float(class_means.min())
        verdict['h1_pass']             = (verdict['h1_mean_pf'] >= thr['h1_mean_pf_min'] and
                                          verdict['h1_min_pf_per_class'] >= thr['h1_min_pf_per_class'])
        verdict['h1_per_class']        = class_means.round(3).to_dict()

# H6 — XGBoost-Lift over LGBM per asset class (only meaningful if XGBoost was trained)
if 'XGBoost' in MODELS_TO_TRAIN:
    # ... (Logik: pro asset_class XGB-PF vs LGBM-PF vergleichen, zählen wie viele über Schwelle)
    verdict['h6_pending'] = 'XGBoost not trained in this MVP run (set MODELS_TO_TRAIN to include XGBoost)'
else:
    verdict['h6_pending'] = 'XGBoost not trained — see MODELS_TO_TRAIN config'

# H5 — Consensus-Lift (only meaningful if all 3 models trained)
if set(MODELS_TO_TRAIN) >= {'LightGBM', 'XGBoost', 'CatBoost'}:
    verdict['h5_pending'] = 'Consensus filter not yet implemented in NB13 MVP — see ANN-004'
else:
    verdict['h5_pending'] = 'Need all 3 models trained for Consensus — set MODELS_TO_TRAIN'

# Architecture-Variante-Hint
if 'h1_pass' in verdict:
    if verdict['h1_pass']:
        verdict['architecture_hint'] = 'Variante A (Universal) plausibel — H1 erfüllt, FX-trained Modell generalisiert'
    elif verdict.get('h1_min_pf_per_class', 0) < 1.0:
        verdict['architecture_hint'] = 'Variante C (Router) wahrscheinlich — mindestens eine Asset-Klasse bricht hart'
    else:
        verdict['architecture_hint'] = 'Variante B (Per-Cluster-Cutoffs) wahrscheinlich — Edge da, aber per-Class kalibriert besser'

print(json.dumps(verdict, indent=2, default=str))

## Section 10 — Result Persistence

Alles nach `/results/nb13/` schreiben. Strukturierte Subordner, RUN_DATE im Filename.

In [ ]:
# 10.1 Cross-Asset Matrix (full table)
matrix_path = RESULTS_DIR / 'metrics' / f'cross_asset_matrix_{RUN_DATE}.csv'
matrix_df.to_csv(matrix_path, index=False)
print(f'[1/5] Matrix table       → {matrix_path.relative_to(PROJECT_ROOT)}')

# 10.2 TF Comparison
tf_path = RESULTS_DIR / 'summaries' / f'tf_comparison_{RUN_DATE}.csv'
tf_summary.to_csv(tf_path, index=False)
print(f'[2/5] TF comparison      → {tf_path.relative_to(PROJECT_ROOT)}')

# 10.3 SHAP
if not shap_df.empty:
    shap_path = RESULTS_DIR / 'shap' / f'shap_per_class_{RUN_DATE}.csv'
    shap_df.to_csv(shap_path, index=False)
    print(f'[3/5] SHAP per class     → {shap_path.relative_to(PROJECT_ROOT)}')
else:
    print(f'[3/5] SHAP skipped (empty)')

# 10.4 Label Balance Report
if not balance_df.empty:
    bal_path = RESULTS_DIR / 'summaries' / f'label_balance_{RUN_DATE}.csv'
    balance_df.to_csv(bal_path, index=False)
    print(f'[4/5] Label balance      → {bal_path.relative_to(PROJECT_ROOT)}')

# 10.5 Full JSON snapshot (Verdict + Run-Config)
snapshot = {
    'notebook':          'NB13_cross_asset_generalization',
    'experiment_id':     EXPERIMENT_ID,
    'run_config':        RUN_CONFIG,
    'verdict':           verdict,
    'cross_asset_matrix': matrix_df.replace([np.inf, -np.inf], 'inf').to_dict(orient='records'),
    'tf_comparison':     tf_summary.to_dict(orient='records'),
    'shap_per_class':    shap_df.to_dict(orient='records') if not shap_df.empty else [],
    'label_balance':     balance_df.to_dict(orient='records') if not balance_df.empty else [],
}
json_path = RESULTS_DIR / 'summaries' / f'nb13_full_snapshot_{RUN_DATE}.json'
with open(json_path, 'w') as f:
    json.dump(snapshot, f, indent=2, default=str)
print(f'[5/5] Full snapshot      → {json_path.relative_to(PROJECT_ROOT)}')

## Section 11 — Final Verdict

Human-readable Conclusion mit konkreten nächsten Schritten.

In [ ]:
print('=' * 70)
print(f'NB13 VERDICT — {RUN_DATE} — commit {GIT_COMMIT}')
print('=' * 70)

if 'h1_pass' in verdict:
    print(f'\nH1 (Universal-Strafe quantifiziert):')
    print(f'  Mean PF across asset classes: {verdict["h1_mean_pf"]:.3f}   threshold: {thr["h1_mean_pf_min"]:.2f}')
    print(f'  Min PF per class:             {verdict["h1_min_pf_per_class"]:.3f}   threshold: {thr["h1_min_pf_per_class"]:.2f}')
    print(f'  Per class:                    {verdict["h1_per_class"]}')
    print(f'  Verdict: H1 {"PASS" if verdict["h1_pass"] else "FAIL"}')

print(f'\nH5 (Consensus generalizes): {verdict.get("h5_pending", "N/A")}')
print(f'H6 (XGBoost-Lift generalizes): {verdict.get("h6_pending", "N/A")}')

print(f'\nArchitecture Hint: {verdict.get("architecture_hint", "insufficient data")}')

print(f'\n——— Next Research Steps ———')
print(f'  1. Prüfen ob H1-Pass / per-class-Bruch konsistent ist mit Erwartung')
print(f'  2. Für H5/H6: Re-run mit MODELS_TO_TRAIN = [LightGBM, XGBoost, CatBoost]')
print(f'  3. NB13b für Indices (wenn Polygon aktiviert)')
print(f'  4. NB14 für deep Multi-TF-Analyse')
print(f'  5. NB15 Architektur-Decision basierend auf NB13 + NB14 Daten')

print('\n' + '=' * 70)
print(f'Results in /results/nb13/ — ready for Section 12 (Auto-Push)')
print('=' * 70)

## Section 12 — Auto-Push Results to GitHub (Optional)

Setup: siehe [/docs/colab_auto_push.md](../docs/colab_auto_push.md). Token `GITHUB_TOKEN` in Colab Secrets ablegen, Notebook-Access aktivieren.

In [ ]:
import shutil
from pathlib import Path as _P

if not IS_COLAB:
    print('Local run — skip auto-push (files already in repo).')
else:
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception as e:
        GH_TOKEN = None
        print(f'ERROR: cannot read GITHUB_TOKEN: {e}')
        print('Setup: see /docs/colab_auto_push.md')

    if GH_TOKEN:
        NB_TAG = 'nb13'
        REPO_URL_HTTPS = 'github.com/ecoNC/pace-algo.git'
        TMP_REPO = _P('/tmp/pace-algo-push')
        if TMP_REPO.exists():
            shutil.rmtree(TMP_REPO)
        subprocess.run(['git', 'clone', '--quiet',
                        f'https://{GH_TOKEN}@{REPO_URL_HTTPS}', str(TMP_REPO)], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.name', 'ecoNC'], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.email',
                        'ecoNC@users.noreply.github.com'], check=True)
        # Copy ALL nb13 results from this run (matching RUN_DATE)
        copied = []
        for f in sorted((RESULTS_DIR).rglob(f'*{RUN_DATE}*')):
            if not f.is_file(): continue
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP_REPO / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(rel)
        # Also copy config snapshot
        for f in sorted((RESULTS_DIR / 'config_snapshots').glob(f'*{EXPERIMENT_ID}*')):
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP_REPO / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(rel)
        print(f'Copied {len(copied)} files:')
        for c in copied:
            print(f'  results/{c}')

        subprocess.run(['git', '-C', str(TMP_REPO), 'pull', '--rebase', '--quiet',
                        'origin', 'main'], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'add', 'results/'], check=True)
        r_status = subprocess.run(['git', '-C', str(TMP_REPO), 'status', '--porcelain'],
                                   capture_output=True, text=True)
        if not r_status.stdout.strip():
            print('Nothing to commit (results already on origin).')
        else:
            msg = f'{NB_TAG.upper()} results: cross-asset run {RUN_DATE} ({len(copied)} files)'
            subprocess.run(['git', '-C', str(TMP_REPO), 'commit', '-m', msg], check=True)
            sha = subprocess.run(['git', '-C', str(TMP_REPO), 'rev-parse', '--short', 'HEAD'],
                                  capture_output=True, text=True).stdout.strip()
            subprocess.run(['git', '-C', str(TMP_REPO), 'push', 'origin', 'main'], check=True)
            print(f'✓ Pushed as ecoNC ({sha})')
            print(f'  https://github.com/ecoNC/pace-algo/commit/{sha}')
        shutil.rmtree(TMP_REPO)